In [0]:
from pyspark.sql import functions as F

REF = "/Volumes/voebem/bronze/arquivos/referencias"

# caractere que NAO existe no arquivo -> desliga o quoting do leitor de CSV
SEM_ASPAS = chr(0)

In [0]:
aerodromos = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("encoding", "ISO-8859-1")   # latin-1, nao UTF-8
    .option("quote", SEM_ASPAS)         # desliga o quoting: aspas aqui sao "segundos"
    .load(f"{REF}/AerodromosPublicos.csv")
)

aerodromos = aerodromos.select(
    F.col("`Código OACI`").alias("icao"),
    F.col("CIAD").alias("ciad"),
    F.col("Nome").alias("nome"),
    F.col("`Município`").alias("municipio"),
    F.col("UF").alias("uf"),
    F.col("`Município Servido`").alias("municipio_servido"),
    F.col("`UF Servido`").alias("uf_servido"),
    F.col("Latitude").alias("latitude"),
    F.col("Longitude").alias("longitude"),
    F.col("Altitude").alias("altitude"),
    F.col("`Situação`").alias("situacao"),
).withColumn("_ingerido_em", F.current_timestamp())

aerodromos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("voebem.bronze.aerodromos")

print(f"bronze.aerodromos: {spark.table('voebem.bronze.aerodromos').count():,} linhas")
display(spark.sql("SELECT icao, nome, municipio, uf FROM voebem.bronze.aerodromos WHERE icao IN ('SBRB','SBGR','SBSP','SBFZ')"))

In [0]:
def ler_empresas(arquivo: str):
    """Le um cadastro de empresas. Sem uniao, sem enriquecimento: uma tabela por arquivo."""
    import csv
    import pandas as pd

    with open(f"{REF}/{arquivo}", 'r', encoding='utf-8') as f:
        first_line = f.readline()

    if 'atividades_aereas' in first_line:
        # Arquivos nacionais: delimitador virgula, header repetido no arquivo
        # Coleta todas as linhas de dados (ignora headers repetidos) e deduplica por id_empresa_aerea
        with open(f"{REF}/{arquivo}", 'r', encoding='utf-8') as f:
            next(f)  # Pula header malformado (delimitador triple-quote)
            reader = csv.reader(f)
            header = None
            data_rows = []
            for row in reader:
                if not row:
                    continue
                if row[0] == 'atividades_aereas':
                    if header is None:
                        header = row
                else:
                    data_rows.append(row)

        pdf = pd.DataFrame(data_rows, columns=header)
        df = spark.createDataFrame(pdf)

        return (
            df.dropDuplicates(["id_empresa_aerea"])
            .select(
                F.col("icao").alias("icao"),
                F.col("iata").alias("sigla_iata"),
                F.col("razao_social").alias("razao_social"),
                F.col("atividades_aereas").alias("servico"),
                F.col("cidade").alias("cidade"),
                F.col("uf").alias("uf"),
                F.col("situacao").alias("situacao"),
            )
            .withColumn("_arquivo_origem", F.lit(arquivo))
            .withColumn("_ingerido_em", F.current_timestamp())
        )
    else:
        # Arquivos estrangeiros: delimitador ponto-e-virgula, formato normal
        return (
            spark.read.format("csv")
            .option("sep", ";")
            .option("header", "true")
            .option("skipRows", 1)
            .option("encoding", "UTF-8")
            .option("quote", '"'
            .load(f"{REF}/{arquivo}")
            .select(
                F.col("ICAO").alias("icao"),
                F.col("Estrangeira").alias("sigla_iata"),
                F.col("Razao").alias("razao_social"),
                F.col("Servico").alias("servico"),
                F.col("Cidade").alias("cidade"),
                F.col("UF").alias("uf"),
                F.col("Ativa").alias("situacao"),
            )
            .withColumn("_arquivo_origem", F.lit(arquivo))
            .withColumn("_ingerido_em", F.current_timestamp())
        )


for arquivo, tabela in [
    ("pda_empresas_aereas_nacionais.csv",    "voebem.bronze.empresas_nacionais"),
    ("pda_empresas_aereas_estrangeiros.csv", "voebem.bronze.empresas_estrangeiras"),
]:
    ler_empresas(arquivo).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(tabela)
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
display(spark.sql("""
    SELECT 'empresas_nacionais' AS tabela, COUNT(*) AS linhas,
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voebem.bronze.empresas_nacionais
    UNION ALL
    SELECT 'empresas_estrangeiras', COUNT(*),
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END)
    FROM voebem.bronze.empresas_estrangeiras
"""))

In [0]:
display(spark.sql("""
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_nacionais
    WHERE icao IN ('GLO','TAM','AZU','PAM')
    ORDER BY icao
"""))

display(spark.sql("""
    SELECT icao, razao_social, servico, situacao
    FROM voebem.bronze.empresas_estrangeiras
    WHERE icao IN ('AAL','TAP','AVA','ARG')
    ORDER BY icao
"""))

In [0]:
CODIGOS = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

codigos = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")
codigos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("voebem.bronze.codigos_operacao")

print(f"bronze.codigos_operacao: {spark.table('voebem.bronze.codigos_operacao').count()} linhas")
display(spark.table("voebem.bronze.codigos_operacao"))

In [0]:
display(spark.sql("SHOW TABLES IN voebem.bronze"))

In [0]:
display(spark.sql("""
    SELECT 'aerodromos' AS tabela,
           COUNT(*) AS linhas,
           COUNT(DISTINCT icao) AS icao_distintos,
           SUM(CASE WHEN icao IS NULL OR icao = '' THEN 1 ELSE 0 END) AS icao_nulos,
           SUM(CASE WHEN nome IS NULL OR nome = '' THEN 1 ELSE 0 END) AS nome_nulos
    FROM voebem.bronze.aerodromos
    UNION ALL
    SELECT 'empresas_nacionais',
           COUNT(*),
           COUNT(DISTINCT icao),
           SUM(CASE WHEN icao IS NULL OR icao = '' THEN 1 ELSE 0 END),
           SUM(CASE WHEN razao_social IS NULL OR razao_social = '' THEN 1 ELSE 0 END)
    FROM voebem.bronze.empresas_nacionais
    UNION ALL
    SELECT 'empresas_estrangeiras',
           COUNT(*),
           COUNT(DISTINCT icao),
           SUM(CASE WHEN icao IS NULL OR icao = '' THEN 1 ELSE 0 END),
           SUM(CASE WHEN razao_social IS NULL OR razao_social = '' THEN 1 ELSE 0 END)
    FROM voebem.bronze.empresas_estrangeiras
    UNION ALL
    SELECT 'codigos_operacao',
           COUNT(*),
           COUNT(DISTINCT codigo),
           0,
           SUM(CASE WHEN descricao IS NULL OR descricao = '' THEN 1 ELSE 0 END)
    FROM voebem.bronze.codigos_operacao
"""))

print("--- Duplicatas de icao em aerodromos ---")
display(spark.sql("""
    SELECT icao, COUNT(*) AS cnt
    FROM voebem.bronze.aerodromos
    GROUP BY icao
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
"""))

print("--- Empresas nacionais: top 10 por situacao ---")
display(spark.sql("""
    SELECT situacao, COUNT(*) AS cnt
    FROM voebem.bronze.empresas_nacionais
    GROUP BY situacao
    ORDER BY cnt DESC
    LIMIT 10
"""))

print("--- Empresas estrangeiras: top 10 por situacao ---")
display(spark.sql("""
    SELECT situacao, COUNT(*) AS cnt
    FROM voebem.bronze.empresas_estrangeiras
    GROUP BY situacao
    ORDER BY cnt DESC
    LIMIT 10
"""))

## Validação da Camada Bronze — Inspeção Completa

Consultas **somente leitura** (sem nenhuma alteração nos dados) para verificar:
1. Quantidade de registros por tabela
2. Estrutura (schema) de cada tabela
3. Primeiros registros de cada tabela

In [0]:
tabelas = [
    "voebem.bronze.aerodromos",
    "voebem.bronze.empresas_nacionais",
    "voebem.bronze.empresas_estrangeiras",
    "voebem.bronze.codigos_operacao",
    "voebem.bronze.vra",
]

for t in tabelas:
    df = spark.table(t)
    print(f"\n{'='*60}")
    print(f"TABELA: {t}")
    print(f"{'='*60}")

    # 1) Quantidade de registros
    total = df.count()
    print(f"\n1) QUANTIDADE DE REGISTROS: {total:,}")

    # 2) Estrutura da tabela (schema)
    print(f"\n2) ESTRUTURA DA TABELA ({len(df.columns)} colunas):")
    for f in df.schema.fields:
        print(f"   {f.name:30s}  {f.dataType.simpleString():15s}  nullable={f.nullable}")

    # 3) Primeiros registros
    print(f"\n3) PRIMEIROS REGISTROS (5):")
    display(df.limit(5))